In [ ]:
import pandas as pd
import torch
import numpy as np
import logging
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

# 提供されたSparseTPRクラスをインポートします
from student import SparseTPR

# ログ設定
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def load_data(feature_path, target_path, device, dtype=torch.float64):
    """CSVファイルからデータを読み込み、PyTorchテンソルに変換する関数"""
    features_df = pd.read_csv(feature_path, header=None)
    target_df = pd.read_csv(target_path, header=None)
    
    # DataFrameをNumPy配列に変換し、PyTorchテンソルに変換
    features_tensor = torch.tensor(features_df.values, dtype=dtype, device=device)
    target_tensor = torch.tensor(target_df.values, dtype=dtype, device=device)
    
    return features_tensor, target_tensor

def main():
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.set_default_dtype(torch.float64)

    # --- 1. 設定 ---
    # データセットのパス
    train_features_path = "datasets/dataset_xu_2024/Protein/split_0/train_features.csv"
    train_target_path   = "datasets/dataset_xu_2024/Protein/split_0/train_target.csv"
    test_features_path  = "datasets/dataset_xu_2024/Protein/split_0/test_features.csv"
    test_target_path    = "datasets/dataset_xu_2024/Protein/split_0/test_target.csv"

    # モデルと学習のハイパーパラメータ
    M = 250             # 誘導点の数（データサイズに応じて調整）
    EPOCHS = 200        # 学習エポック数
    BATCH_SIZE = 1024    # バッチサイズ
    HYPER_LR = 0.01     # ハイパーパラメータの学習率
    VAR_LR = 0.1        # 変分パラメータの学習率（Polyak Averagingの係数）

    # --- 2. データの読み込みと準備 ---
    X_train, y_train = load_data(train_features_path, train_target_path, 'cuda')
    X_test, y_test = load_data(test_features_path, test_target_path, 'cuda')

    print(f'Train features shape: {X_train.shape}, Train target shape: {y_train.shape}')
    print(f'Test features shape:  {X_test.shape}, Test target shape: {y_test.shape}')

    logging.info(f"Train data shape: X={X_train.shape}, y={y_train.shape}")
    logging.info(f"Test data shape:  X={X_test.shape}, y={y_test.shape}")


    hyper_settings = {
        'lengthscale': {"optim": "MLE"},
        'outputscale': {"optim": "FIX", "init": 1.0},
        'dof_func': {"optim": "MLE"},  # High, fixed DoF for near-GP behavior},
        'dof_lik':  {"optim": "MLE"},  # High, fixed DoF for near-GP behavior},
        'noisescale': {"optim": "MLE"}
    }
    hyper_settings = {
        'lengthscale': {"optim": "MAP"},
        'outputscale': {"optim": "FIX", "init": 1.0},
        'dof_func':    {"optim": "MAP"},  # High, fixed DoF for near-GP behavior},
        'dof_lik':     {"optim": "MAP"},  # High, fixed DoF for near-GP behavior},
        'noisescale':  {"optim": "MAP"}
    }

    # --- 3. モデルの初期化 ---
    model = SparseTPR(
        X=X_train,
        y=y_train,
        M=M,
        kernel="rbf",  # RBFカーネルを使用
        inducing_init_method="kmeans",  # k-meansで誘導点を初期化
        device='cuda',
        hyper_settings=hyper_settings
    )

    # --- 4. モデルの学習 ---
    history = model.fit(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        hyper_lr=HYPER_LR,
        var_lr=VAR_LR,
        X_test=X_test,
        y_test=y_test,
        eval_interval=10 # 10エポックごとにテストデータで評価
    )

    # --- 5. 予測と評価 ---
    model.eval() # 評価モードに切り替え
    with torch.no_grad():
        # テストデータで予測
        pred_dist = model.predict(X_test)
        
        # 予測分布の平均値を取得
        y_pred_mean = pred_dist['loc']
        
        # テンソルをCPU上のNumPy配列に変換
        y_pred_numpy = y_pred_mean.cpu().numpy()
        y_test_numpy = y_test.cpu().numpy().squeeze()

        # RMSE (Root Mean Squared Error) を計算
        rmse = np.sqrt(mean_squared_error(y_test_numpy, y_pred_numpy))
        logging.info(f"Final Test RMSE: {rmse:.4f}")


main()

2025-09-11 23:35:35,640 - INFO - Train data shape: X=torch.Size([36584, 9]), y=torch.Size([36584, 1])
2025-09-11 23:35:35,641 - INFO - Test data shape:  X=torch.Size([9146, 9]), y=torch.Size([9146, 1])
2025-09-11 23:35:35,642 - INFO - Sampled initial lengthscale (Optim mode: MLE): [0.54252203 0.48043177 0.20353498 0.39563664 0.58230396 0.52125128
 0.82028681 0.1249347  0.91013377]
2025-09-11 23:35:35,643 - INFO - Using provided initial outputscale (Optim mode: FIX): 1.0
2025-09-11 23:35:35,644 - INFO - Sampled initial dof_func (Optim mode: MLE): 6.154538461249912
2025-09-11 23:35:35,645 - INFO - Sampled initial dof_lik (Optim mode: MLE): 2.0
2025-09-11 23:35:35,645 - INFO - Sampled initial noisescale (Optim mode: MLE): 1.1920434818338546


Train features shape: torch.Size([36584, 9]), Train target shape: torch.Size([36584, 1])
Test features shape:  torch.Size([9146, 9]), Test target shape: torch.Size([9146, 1])


2025-09-11 23:35:36,099 - INFO - Starting SVI optimization for 200 epochs...
2025-09-11 23:35:49,057 - INFO - Epoch  10/200 | ELBO: -124962.90 | l: [25.001, 23.666, 6.875, 17.021, 28.361, 26.832, 47.755, 5.303, 46.716] | var: 1.000 | noise2: 3.861325 | dof_func: 0.32 | dof_lik: 1.96
2025-09-11 23:35:49,211 - INFO - Epoch  10/200 | Test Metrics: RMSE: 2.486
2025-09-11 23:36:02,130 - INFO - Epoch  20/200 | ELBO: -106678.78 | l: [100.385, 112.701, 28.528, 66.968, 108.916, 88.904, 181.744, 21.856, 181.332] | var: 1.000 | noise2: 19.022582 | dof_func: 0.11 | dof_lik: 8.13
2025-09-11 23:36:02,284 - INFO - Epoch  20/200 | Test Metrics: RMSE: 1.180
2025-09-11 23:36:15,342 - INFO - Epoch  30/200 | ELBO: -111092.62 | l: [202.084, 239.455, 53.698, 130.831, 238.198, 197.637, 406.282, 43.873, 487.644] | var: 1.000 | noise2: 24.755663 | dof_func: 0.09 | dof_lik: 18.57
2025-09-11 23:36:15,498 - INFO - Epoch  30/200 | Test Metrics: RMSE: 1.172
2025-09-11 23:36:28,433 - INFO - Epoch  40/200 | ELBO: -11

In [ ]:
import pandas as pd
import torch
import numpy as np
import logging
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

# 提供されたSparseTPRクラスをインポートします
from student import SparseTPR

# ログ設定
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def load_data(feature_path, target_path, device, dtype=torch.float64):
    """CSVファイルからデータを読み込み、PyTorchテンソルに変換する関数"""
    features_df = pd.read_csv(feature_path, header=None)
    target_df = pd.read_csv(target_path, header=None)
    
    # DataFrameをNumPy配列に変換し、PyTorchテンソルに変換
    features_tensor = torch.tensor(features_df.values, dtype=dtype, device=device)
    target_tensor = torch.tensor(target_df.values, dtype=dtype, device=device)
    
    return features_tensor, target_tensor

def main():
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.set_default_dtype(torch.float64)

    # --- 1. 設定 ---
    # データセットのパス
    train_features_path = "datasets/dataset_xu_2024/Protein/split_0/train_features.csv"
    train_target_path   = "datasets/dataset_xu_2024/Protein/split_0/train_target.csv"
    test_features_path  = "datasets/dataset_xu_2024/Protein/split_0/test_features.csv"
    test_target_path    = "datasets/dataset_xu_2024/Protein/split_0/test_target.csv"

    # モデルと学習のハイパーパラメータ
    M = 250             # 誘導点の数（データサイズに応じて調整）
    EPOCHS = 200        # 学習エポック数
    BATCH_SIZE = 1024    # バッチサイズ
    HYPER_LR = 0.01     # ハイパーパラメータの学習率
    VAR_LR = 0.1        # 変分パラメータの学習率（Polyak Averagingの係数）

    # --- 2. データの読み込みと準備 ---
    X_train, y_train = load_data(train_features_path, train_target_path, 'cuda')
    X_test, y_test = load_data(test_features_path, test_target_path, 'cuda')

    print(f'Train features shape: {X_train.shape}, Train target shape: {y_train.shape}')
    print(f'Test features shape:  {X_test.shape}, Test target shape: {y_test.shape}')

    logging.info(f"Train data shape: X={X_train.shape}, y={y_train.shape}")
    logging.info(f"Test data shape:  X={X_test.shape}, y={y_test.shape}")


    hyper_settings = {
        'lengthscale': {"optim": "MLE"},
        'outputscale': {"optim": "FIX", "init": 1.0},
        'dof_func': {"optim": "MLE"},  # High, fixed DoF for near-GP behavior},
        'dof_lik':  {"optim": "MLE"},  # High, fixed DoF for near-GP behavior},
        'noisescale': {"optim": "MLE"}
    }
    hyper_settings = {
        'lengthscale': {"optim": "MAP"},
        'outputscale': {"optim": "FIX", "init": 1.0},
        'dof_func':    {"optim": "MAP"},  # High, fixed DoF for near-GP behavior},
        'dof_lik':     {"optim": "MAP"},  # High, fixed DoF for near-GP behavior},
        'noisescale':  {"optim": "MAP"}
    }

    # --- 3. モデルの初期化 ---
    model = SparseTPR(
        X=X_train,
        y=y_train,
        M=M,
        kernel="rbf",  # RBFカーネルを使用
        inducing_init_method="kmeans",  # k-meansで誘導点を初期化
        device='cuda',
        hyper_settings=hyper_settings
    )

    # --- 4. モデルの学習 ---
    history = model.fit(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        hyper_lr=HYPER_LR,
        var_lr=VAR_LR,
        X_test=X_test,
        y_test=y_test,
        eval_interval=10 # 10エポックごとにテストデータで評価
    )

    # --- 5. 予測と評価 ---
    model.eval() # 評価モードに切り替え
    with torch.no_grad():
        # テストデータで予測
        pred_dist = model.predict(X_test)
        
        # 予測分布の平均値を取得
        y_pred_mean = pred_dist['loc']
        
        # テンソルをCPU上のNumPy配列に変換
        y_pred_numpy = y_pred_mean.cpu().numpy()
        y_test_numpy = y_test.cpu().numpy().squeeze()

        # RMSE (Root Mean Squared Error) を計算
        rmse = np.sqrt(mean_squared_error(y_test_numpy, y_pred_numpy))
        logging.info(f"Final Test RMSE: {rmse:.4f}")


main()

2025-09-11 23:43:04,023 - INFO - Train data shape: X=torch.Size([36584, 9]), y=torch.Size([36584, 1])
2025-09-11 23:43:04,023 - INFO - Test data shape:  X=torch.Size([9146, 9]), y=torch.Size([9146, 1])
2025-09-11 23:43:04,025 - INFO - Sampled initial lengthscale (Optim mode: MAP): [0.54252203 0.48043177 0.20353498 0.39563664 0.58230396 0.52125128
 0.82028681 0.1249347  0.91013377]
2025-09-11 23:43:04,026 - INFO - Using provided initial outputscale (Optim mode: FIX): 1.0
2025-09-11 23:43:04,027 - INFO - Sampled initial dof_func (Optim mode: MAP): 6.154538461249912
2025-09-11 23:43:04,027 - INFO - Sampled initial dof_lik (Optim mode: MAP): 2.0
2025-09-11 23:43:04,028 - INFO - Sampled initial noisescale (Optim mode: MAP): 1.1920434818338546


Train features shape: torch.Size([36584, 9]), Train target shape: torch.Size([36584, 1])
Test features shape:  torch.Size([9146, 9]), Test target shape: torch.Size([9146, 1])


2025-09-11 23:43:04,489 - INFO - Starting SVI optimization for 200 epochs...
2025-09-11 23:43:18,150 - INFO - Epoch  10/200 | ELBO: -123661.75 | l: [24.722, 23.274, 6.840, 17.020, 27.975, 26.791, 46.243, 5.260, 45.829] | var: 1.000 | noise2: 3.840089 | dof_func: 0.40 | dof_lik: 1.94
2025-09-11 23:43:18,305 - INFO - Epoch  10/200 | Test Metrics: RMSE: 3.432
2025-09-11 23:43:31,693 - INFO - Epoch  20/200 | ELBO: -109345.21 | l: [78.299, 86.705, 28.144, 57.731, 82.364, 71.934, 102.055, 22.207, 107.797] | var: 1.000 | noise2: 18.476035 | dof_func: 0.24 | dof_lik: 7.93
2025-09-11 23:43:31,848 - INFO - Epoch  20/200 | Test Metrics: RMSE: 1.236
2025-09-11 23:43:45,587 - INFO - Epoch  30/200 | ELBO: -108859.36 | l: [105.590, 113.111, 51.523, 88.943, 104.645, 92.060, 114.551, 44.432, 109.194] | var: 1.000 | noise2: 21.955947 | dof_func: 0.23 | dof_lik: 17.26
2025-09-11 23:43:45,743 - INFO - Epoch  30/200 | Test Metrics: RMSE: 1.181
2025-09-11 23:43:59,515 - INFO - Epoch  40/200 | ELBO: -111577.

In [ ]:
import pandas as pd
import torch
import numpy as np
import logging
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

# 提供されたSparseTPRクラスをインポートします
from student import SparseTPR

# ログ設定
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def load_data(feature_path, target_path, device, dtype=torch.float64):
    """CSVファイルからデータを読み込み、PyTorchテンソルに変換する関数"""
    features_df = pd.read_csv(feature_path, header=None)
    target_df = pd.read_csv(target_path, header=None)
    
    # DataFrameをNumPy配列に変換し、PyTorchテンソルに変換
    features_tensor = torch.tensor(features_df.values, dtype=dtype, device=device)
    target_tensor = torch.tensor(target_df.values, dtype=dtype, device=device)
    
    return features_tensor, target_tensor

def main():
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.set_default_dtype(torch.float64)

    # --- 1. 設定 ---
    # データセットのパス
    train_features_path = "datasets/dataset_xu_2024/Elevators/split_0/train_features.csv"
    train_target_path   = "datasets/dataset_xu_2024/Elevators/split_0/train_target.csv"
    test_features_path  = "datasets/dataset_xu_2024/Elevators/split_0/test_features.csv"
    test_target_path    = "datasets/dataset_xu_2024/Elevators/split_0/test_target.csv"

    # モデルと学習のハイパーパラメータ
    M = 250             # 誘導点の数（データサイズに応じて調整）
    EPOCHS = 200        # 学習エポック数
    BATCH_SIZE = 1024    # バッチサイズ
    HYPER_LR = 0.01     # ハイパーパラメータの学習率
    VAR_LR = 0.1        # 変分パラメータの学習率（Polyak Averagingの係数）

    # --- 2. データの読み込みと準備 ---
    X_train, y_train = load_data(train_features_path, train_target_path, 'cuda')
    X_test, y_test = load_data(test_features_path, test_target_path, 'cuda')

    print(f'Train features shape: {X_train.shape}, Train target shape: {y_train.shape}')
    print(f'Test features shape:  {X_test.shape}, Test target shape: {y_test.shape}')

    logging.info(f"Train data shape: X={X_train.shape}, y={y_train.shape}")
    logging.info(f"Test data shape:  X={X_test.shape}, y={y_test.shape}")


    hyper_settings = {
        'lengthscale': {"optim": "MLE"},
        'outputscale': {"optim": "FIX", "init": 1.0},
        'dof_func': {"optim": "MLE"},  # High, fixed DoF for near-GP behavior},
        'dof_lik':  {"optim": "MLE"},  # High, fixed DoF for near-GP behavior},
        'noisescale': {"optim": "MLE"}
    }
    hyper_settings = {
        'lengthscale': {"optim": "MAP"},
        'outputscale': {"optim": "FIX", "init": 1.0},
        'dof_func':    {"optim": "MAP"},  # High, fixed DoF for near-GP behavior},
        'dof_lik':     {"optim": "MAP"},  # High, fixed DoF for near-GP behavior},
        'noisescale':  {"optim": "MAP"}
    }

    # --- 3. モデルの初期化 ---
    model = SparseTPR(
        X=X_train,
        y=y_train,
        M=M,
        kernel="rbf",  # RBFカーネルを使用
        inducing_init_method="kmeans",  # k-meansで誘導点を初期化
        device='cuda',
        hyper_settings=hyper_settings
    )

    # --- 4. モデルの学習 ---
    history = model.fit(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        hyper_lr=HYPER_LR,
        var_lr=VAR_LR,
        X_test=X_test,
        y_test=y_test,
        eval_interval=10 # 10エポックごとにテストデータで評価
    )

    # --- 5. 予測と評価 ---
    model.eval() # 評価モードに切り替え
    with torch.no_grad():
        # テストデータで予測
        pred_dist = model.predict(X_test)
        
        # 予測分布の平均値を取得
        y_pred_mean = pred_dist['loc']
        
        # テンソルをCPU上のNumPy配列に変換
        y_pred_numpy = y_pred_mean.cpu().numpy()
        y_test_numpy = y_test.cpu().numpy().squeeze()

        # RMSE (Root Mean Squared Error) を計算
        rmse = np.sqrt(mean_squared_error(y_test_numpy, y_pred_numpy))
        logging.info(f"Final Test RMSE: {rmse:.4f}")


main()

2025-09-11 23:50:09,479 - INFO - Train data shape: X=torch.Size([13279, 18]), y=torch.Size([13279, 1])
2025-09-11 23:50:09,480 - INFO - Test data shape:  X=torch.Size([3320, 18]), y=torch.Size([3320, 1])
2025-09-11 23:50:09,482 - INFO - Sampled initial lengthscale (Optim mode: MAP): [0.54252203 0.48043177 0.20353498 0.39563664 0.58230396 0.52125128
 0.82028681 0.1249347  0.91013377 0.70601814 0.923547   0.6586284
 0.52555166 0.45990613 0.43100035 0.60363698 0.48167484 0.72349709]
2025-09-11 23:50:09,483 - INFO - Using provided initial outputscale (Optim mode: FIX): 1.0
2025-09-11 23:50:09,504 - INFO - Sampled initial dof_func (Optim mode: MAP): 3.1574972609344925
2025-09-11 23:50:09,505 - INFO - Sampled initial dof_lik (Optim mode: MAP): 2.205834907306894
2025-09-11 23:50:09,506 - INFO - Sampled initial noisescale (Optim mode: MAP): 0.9107284790116058


Train features shape: torch.Size([13279, 18]), Train target shape: torch.Size([13279, 1])
Test features shape:  torch.Size([3320, 18]), Test target shape: torch.Size([3320, 1])


2025-09-11 23:50:10,821 - INFO - Starting SVI optimization for 200 epochs...
2025-09-11 23:50:16,500 - INFO - Epoch  10/200 | ELBO: -22238.74 | l: [3.386, 3.034, 1.317, 2.552, 3.605, 3.303, 5.090, 0.815, 5.733, 4.183, 4.495, 4.060, 3.412, 3.409, 0.744, 1.915, 0.732, 4.221] | var: 1.000 | noise2: 2.111951 | dof_func: 1.19 | dof_lik: 2.96
2025-09-11 23:50:16,532 - INFO - Epoch  10/200 | Test Metrics: RMSE: 1.275
2025-09-11 23:50:21,565 - INFO - Epoch  20/200 | ELBO: -13209.78 | l: [6.960, 6.518, 3.090, 5.961, 7.360, 6.792, 10.783, 2.213, 14.320, 9.134, 9.226, 8.721, 7.678, 9.466, 2.486, 10.447, 2.269, 8.797] | var: 1.000 | noise2: 0.702100 | dof_func: 0.87 | dof_lik: 7.23
2025-09-11 23:50:21,594 - INFO - Epoch  20/200 | Test Metrics: RMSE: 1.327
2025-09-11 23:50:26,561 - INFO - Epoch  30/200 | ELBO: -11248.97 | l: [21.671, 20.193, 10.163, 19.944, 23.763, 25.107, 37.104, 8.650, 50.471, 32.041, 30.423, 28.907, 26.121, 35.897, 10.651, 44.637, 10.337, 29.314] | var: 1.000 | noise2: 0.275039 

In [ ]:
import pandas as pd
import torch
import numpy as np
import logging
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

# 提供されたSparseTPRクラスをインポートします
from student import SparseTPR

# ログ設定
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def load_data(feature_path, target_path, device, dtype=torch.float64):
    """CSVファイルからデータを読み込み、PyTorchテンソルに変換する関数"""
    features_df = pd.read_csv(feature_path, header=None)
    target_df = pd.read_csv(target_path, header=None)
    
    # DataFrameをNumPy配列に変換し、PyTorchテンソルに変換
    features_tensor = torch.tensor(features_df.values, dtype=dtype, device=device)
    target_tensor = torch.tensor(target_df.values, dtype=dtype, device=device)
    
    return features_tensor, target_tensor

def main():
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    torch.set_default_dtype(torch.float64)

    # --- 1. 設定 ---
    # データセットのパス
    train_features_path = "datasets/dataset_xu_2024/Elevators/split_1/train_features.csv"
    train_target_path   = "datasets/dataset_xu_2024/Elevators/split_1/train_target.csv"
    test_features_path  = "datasets/dataset_xu_2024/Elevators/split_1/test_features.csv"
    test_target_path    = "datasets/dataset_xu_2024/Elevators/split_1/test_target.csv"

    # モデルと学習のハイパーパラメータ
    M = 250             # 誘導点の数（データサイズに応じて調整）
    EPOCHS = 20        # 学習エポック数
    BATCH_SIZE = 1024    # バッチサイズ
    HYPER_LR = 0.01     # ハイパーパラメータの学習率
    VAR_LR = 0.1        # 変分パラメータの学習率（Polyak Averagingの係数）

    # --- 2. データの読み込みと準備 ---
    X_train, y_train = load_data(train_features_path, train_target_path, 'cuda')
    X_test, y_test = load_data(test_features_path, test_target_path, 'cuda')

    print(f'Train features shape: {X_train.shape}, Train target shape: {y_train.shape}')
    print(f'Test features shape:  {X_test.shape}, Test target shape: {y_test.shape}')

    logging.info(f"Train data shape: X={X_train.shape}, y={y_train.shape}")
    logging.info(f"Test data shape:  X={X_test.shape}, y={y_test.shape}")


    hyper_settings = {
        'lengthscale': {"optim": "MLE"},
        'outputscale': {"optim": "FIX", "init": 1.0},
        'dof_func': {"optim": "MLE"},  # High, fixed DoF for near-GP behavior},
        'dof_lik':  {"optim": "MLE"},  # High, fixed DoF for near-GP behavior},
        'noisescale': {"optim": "MLE"}
    }
    hyper_settings = {
        'lengthscale': {"optim": "MAP"},
        'outputscale': {"optim": "FIX", "init": 1.0},
        'dof_func':    {"optim": "MAP"},  # High, fixed DoF for near-GP behavior},
        'dof_lik':     {"optim": "MAP"},  # High, fixed DoF for near-GP behavior},
        'noisescale':  {"optim": "MAP"}
    }

    # --- 3. モデルの初期化 ---
    model = SparseTPR(
        X=X_train,
        y=y_train,
        M=M,
        kernel="rbf",  # RBFカーネルを使用
        inducing_init_method="kmeans",  # k-meansで誘導点を初期化
        device='cuda',
        hyper_settings=hyper_settings
    )

    # --- 4. モデルの学習 ---
    history = model.fit(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        hyper_lr=HYPER_LR,
        var_lr=VAR_LR,
        X_test=X_test,
        y_test=y_test,
        eval_interval=1 # 10エポックごとにテストデータで評価
    )

    # --- 5. 予測と評価 ---
    model.eval() # 評価モードに切り替え
    with torch.no_grad():
        # テストデータで予測
        pred_dist = model.predict(X_test)
        
        # 予測分布の平均値を取得
        y_pred_mean = pred_dist['loc']
        
        # テンソルをCPU上のNumPy配列に変換
        y_pred_numpy = y_pred_mean.cpu().numpy()
        y_test_numpy = y_test.cpu().numpy().squeeze()

        # RMSE (Root Mean Squared Error) を計算
        rmse = np.sqrt(mean_squared_error(y_test_numpy, y_pred_numpy))
        logging.info(f"Final Test RMSE: {rmse:.4f}")


main()

2025-09-11 23:59:33,579 - INFO - Train data shape: X=torch.Size([13279, 18]), y=torch.Size([13279, 1])
2025-09-11 23:59:33,580 - INFO - Test data shape:  X=torch.Size([3320, 18]), y=torch.Size([3320, 1])
2025-09-11 23:59:33,581 - INFO - Sampled initial lengthscale (Optim mode: MAP): [0.54252203 0.48043177 0.20353498 0.39563664 0.58230396 0.52125128
 0.82028681 0.1249347  0.91013377 0.70601814 0.923547   0.6586284
 0.52555166 0.45990613 0.43100035 0.60363698 0.48167484 0.72349709]
2025-09-11 23:59:33,582 - INFO - Using provided initial outputscale (Optim mode: FIX): 1.0
2025-09-11 23:59:33,583 - INFO - Sampled initial dof_func (Optim mode: MAP): 3.1574972609344925
2025-09-11 23:59:33,584 - INFO - Sampled initial dof_lik (Optim mode: MAP): 2.205834907306894
2025-09-11 23:59:33,584 - INFO - Sampled initial noisescale (Optim mode: MAP): 0.9107284790116058


Train features shape: torch.Size([13279, 18]), Train target shape: torch.Size([13279, 1])
Test features shape:  torch.Size([3320, 18]), Test target shape: torch.Size([3320, 1])


2025-09-11 23:59:33,873 - INFO - Starting SVI optimization for 20 epochs...
2025-09-11 23:59:34,385 - INFO - Epoch   1/20 | ELBO: -36848.41 | l: [0.616, 0.546, 0.232, 0.449, 0.660, 0.592, 0.902, 0.142, 0.931, 0.735, 0.844, 0.701, 0.580, 0.517, 0.380, 0.534, 0.424, 0.745] | var: 1.000 | noise2: 1.035326 | dof_func: 2.77 | dof_lik: 2.36
2025-09-11 23:59:34,392 - INFO - Epoch   1/20 | Test Metrics: RMSE: 1.036
2025-09-11 23:59:34,873 - INFO - Epoch   2/20 | ELBO: -40490.90 | l: [0.714, 0.633, 0.268, 0.520, 0.764, 0.686, 1.045, 0.165, 1.069, 0.847, 0.922, 0.811, 0.672, 0.595, 0.347, 0.494, 0.381, 0.859] | var: 1.000 | noise2: 1.188633 | dof_func: 2.41 | dof_lik: 2.20
2025-09-11 23:59:34,879 - INFO - Epoch   2/20 | Test Metrics: RMSE: 1.028
2025-09-11 23:59:35,393 - INFO - Epoch   3/20 | ELBO: -42616.09 | l: [0.845, 0.748, 0.317, 0.616, 0.905, 0.813, 1.237, 0.195, 1.265, 1.005, 1.090, 0.962, 0.797, 0.700, 0.344, 0.499, 0.358, 1.020] | var: 1.000 | noise2: 1.364811 | dof_func: 2.09 | dof_lik

In [ ]:
import logging
import time
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error
from torch.utils.data import DataLoader, TensorDataset
from linear_operator.operators import to_linear_operator

from student.constants import EPSILON, JITTER
from student.kernels import rbf_kernel, matern52_kernel
from student.priors import GammaPrior, LogNormalPrior
from student.utils import (
    kl_gamma,
    kl_gaussian_gamma_covariance_param,
    get_optimal_gaussian_gamma,
    gaussian_gamma_standard_to_natural_covariance_param,
    gaussian_gamma_natural_to_standard_covariance_param
)


class SparseTPR(nn.Module):
    """
    Sparse Student-t Process Regression (TPR) with a Student-t Likelihood.
    The model is trained using Structured Stochastic Variational Inference (SVI).
    """
    def __init__(
        self,
        X, y, M,
        hyper_settings=None,
        kernel="rbf",
        inducing_init_method="kmeans",
        device=None
    ):
        super().__init__()

        if device is None:
            self.device = X.device if isinstance(X, torch.Tensor) else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        else:
            self.device = torch.device(device)

        # Register data as non-trainable buffers (y is continuous)
        self.register_buffer('X_full', X.to(self.device))
        self.register_buffer('y_full', y.view(-1, 1).to(self.device))

        if self.X_full.ndim == 1: self.X_full = self.X_full.unsqueeze(1)
        if self.y_full.ndim == 1: self.y_full = self.y_full.unsqueeze(1)

        self.N, self.D = self.X_full.shape
        self.M = M
        dtype = self.X_full.dtype

        # Priors for hyperparameters
        self.lengthscale_prior = GammaPrior(3.0, 6.0)
        self.outputscale_prior = GammaPrior(2.0, 0.15)
        self.nu_prior = LogNormalPrior(loc=1.0, scale=1.0)
        self.noise_prior = LogNormalPrior(loc=0.1, scale=0.5)

        # Initialize hyperparameters
        hyperparameters = self._initialize_hyperparameters(hyper_settings)
        lengthscale = hyperparameters['lengthscale']
        outputscale = hyperparameters['outputscale']
        dof_func = hyperparameters['dof_func']
        dof_lik = hyperparameters['dof_lik']
        noisescale = hyperparameters['noisescale']

        # Register hyperparameters as learnable parameters
        self.log_lengthscale = nn.Parameter(torch.log(lengthscale))
        self.log_outputscale = nn.Parameter(torch.log(outputscale))
        self.log_dof_func = nn.Parameter(torch.log(dof_func))
        self.log_dof_lik = nn.Parameter(torch.log(dof_lik))
        self.log_noisescale = nn.Parameter(torch.log(noisescale))
        
        # Initialize inducing points as learnable parameters
        self.Z = nn.Parameter(self._initialize_inducing_points(method=inducing_init_method))

        # Register variational parameters for q(u, r) ~ Normal-Gamma(m_u, S_u, alpha_r, beta_r)
        self.register_buffer('m_u', torch.zeros(self.M, 1, dtype=dtype))
        self.register_buffer('S_u', torch.eye(self.M, dtype=dtype))
        self.register_buffer('alpha_r', dof_func / 2.0)
        self.register_buffer('beta_r', dof_func / 2.0)

        # Set kernel function
        if kernel in (None, "rbf"): self.kernel = rbf_kernel
        elif kernel == "matern52": self.kernel = matern52_kernel
        else:
            logging.info("Unknown kernel specified. Defaulting to RBF kernel.")
            self.kernel = rbf_kernel

        self.to(self.device)

    def _initialize_hyperparameters(self, hyper_settings=None):
        self.hyper_optim_mode = {}
        dtype = self.X_full.dtype
        
        param_configs = {
            'lengthscale': {'prior': self.lengthscale_prior, 'is_vector': True},
            'outputscale': {'prior': self.outputscale_prior, 'is_vector': False},
            'dof_func': {'prior': self.nu_prior, 'is_vector': False},
            'dof_lik': {'prior': self.nu_prior, 'is_vector': False},
            'noisescale': {'prior': self.noise_prior, 'is_vector': False}
        }
        
        initialized_params = {}

        for name, config in param_configs.items():
            settings = (hyper_settings or {}).get(name, {})
            mode = settings.get("optim", "MLE")
            init_val = settings.get("init", None)

            if mode not in ['MLE', 'MAP', 'FIX']: raise ValueError(f"Invalid mode '{mode}' for '{name}'.")
            self.hyper_optim_mode[name] = mode

            if init_val is None:
                sample_shape = (self.D,) if config['is_vector'] else torch.Size([])
                final_value = config['prior'].sample(sample_shape=sample_shape).to(self.device, dtype=dtype)
                if name.startswith('dof'): final_value = final_value.clamp(min=2.0)
                logging.info(f"Sampled initial {name} (Optim mode: {mode}): {final_value.cpu().numpy()}")
            else:
                final_value = init_val
                logging.info(f"Using provided initial {name} (Optim mode: {mode}): {final_value}")
            
            initialized_params[name] = torch.as_tensor(final_value, dtype=dtype, device=self.device)

        ls = initialized_params['lengthscale']
        if ls.ndim == 0: ls = ls.repeat(self.D)
        if ls.shape[0] != self.D: raise ValueError("lengthscale must be scalar or vector of length D")
        initialized_params['lengthscale'] = ls
        
        return initialized_params

    def _initialize_inducing_points(self, method="kmeans"):
        if self.N >= self.M:
            if method == "kmeans":
                X_np = self.X_full.cpu().numpy()
                kmeans = KMeans(n_clusters=self.M, random_state=42, n_init='auto').fit(X_np)
                Z_init = torch.from_numpy(kmeans.cluster_centers_)
            elif method == "random":
                indices = np.random.choice(self.N, self.M, replace=False)
                Z_init = self.X_full[indices].clone()
            else: raise ValueError(f"Unknown init method: {method}")
        else:
            indices = np.random.choice(self.N, self.M, replace=True)
            Z_init = self.X_full[indices].clone()
        return Z_init.to(dtype=self.X_full.dtype, device=self.device)

    def _get_hyperparams(self):
        return {
            "lengthscale": torch.exp(self.log_lengthscale).clamp(min=EPSILON),
            "outputscale": torch.exp(self.log_outputscale).clamp(min=EPSILON),
            "dof_func": torch.exp(self.log_dof_func).clamp(min=EPSILON),
            "dof_lik": torch.exp(self.log_dof_lik).clamp(min=EPSILON),
            "noisescale": torch.exp(self.log_noisescale).clamp(min=EPSILON),
        }

    def _calculate_log_prior(self, params):
        log_prior = torch.tensor(0.0, device=self.device, dtype=params['lengthscale'].dtype)
        if self.hyper_optim_mode['lengthscale'] == 'MAP':
            log_prior += self.lengthscale_prior.log_prob(params['lengthscale']).sum()
        if self.hyper_optim_mode['outputscale'] == 'MAP':
            log_prior += self.outputscale_prior.log_prob(params['outputscale'])
        if self.hyper_optim_mode['dof_func'] == 'MAP':
            log_prior += self.nu_prior.log_prob(params['dof_func'])
        if self.hyper_optim_mode['dof_lik'] == 'MAP':
            log_prior += self.nu_prior.log_prob(params['dof_lik'])
        if self.hyper_optim_mode['noisescale'] == 'MAP':
            log_prior += self.noise_prior.log_prob(params['noisescale'])
        return log_prior

    def _calculate_elbo(self, X_batch, y_batch, K_XZ_batch, K_ZZ, local_params):
        alpha_lambda_batch, beta_lambda_batch, E_lambda_batch = local_params
        params = self._get_hyperparams()
        batch_size = X_batch.shape[0]
        scaling_factor = self.N / batch_size
        
        # --- 1. Expected Log Likelihood Term ---
        E_log_lambda_batch = torch.digamma(alpha_lambda_batch) - torch.log(beta_lambda_batch.clamp(min=EPSILON))
        
        A = K_XZ_batch @ K_ZZ.solve(torch.eye(self.M, device=self.device))
        mu_f_batch = A @ self.m_u
        E_r_inv = self.beta_r / (self.alpha_r - 1.0).clamp(min=EPSILON)
        K_tilde_diag = params['outputscale'] - torch.sum(A * K_XZ_batch, dim=1)
        var_f_batch = E_r_inv * (K_tilde_diag + torch.sum((A @ self.S_u) * A, dim=1))
        E_sq_err = (y_batch.squeeze() - mu_f_batch.squeeze())**2 + var_f_batch

        log_2pi = torch.log(torch.tensor(2 * torch.pi, device=self.device))
        log_lik_per_item = -0.5 * log_2pi - 0.5 * torch.log(params['noisescale']) + 0.5 * E_log_lambda_batch - 0.5 * E_lambda_batch * E_sq_err / params['noisescale']
        exp_log_lik = torch.sum(log_lik_per_item) * scaling_factor

        # --- 2. KL Divergence KL(q(lambda) || p(lambda)) ---
        p_alpha_lambda, p_beta_lambda = params['dof_lik'] / 2.0, params['dof_lik'] / 2.0
        kl_lambda = kl_gamma(alpha_lambda_batch, beta_lambda_batch, p_alpha_lambda, p_beta_lambda).sum() * scaling_factor

        # --- 3. KL Divergence KL(q(u,r) || p(u,r)) ---
        p_alpha_r, p_beta_r = params['dof_func'] / 2.0, params['dof_func'] / 2.0
        prior_mean_u = torch.zeros_like(self.m_u.squeeze())
        kl_u_r = kl_gaussian_gamma_covariance_param(
            mu_q=self.m_u.squeeze(), S_q=self.S_u, alpha_q=self.alpha_r, beta_q=self.beta_r,
            mu_p=prior_mean_u, K_p=K_ZZ.to_dense(), alpha_p=p_alpha_r, beta_p=p_beta_r
        )
        
        return exp_log_lik - kl_lambda - kl_u_r

    def _e_step_local(self, X_batch, y_batch, K_XZ_batch, K_ZZ, params):
        """E-Step for local parameters q(lambda_i) for the mini-batch."""
        with torch.no_grad():
            A_batch = K_XZ_batch @ K_ZZ.solve(torch.eye(self.M, device=self.device, dtype=X_batch.dtype))
            
            # --- Calculate E[(y_i - f_i)^2] ---
            # 1. E[f_i]
            mu_f_batch = A_batch @ self.m_u
            
            # 2. Var(f_i)
            E_r_inv = self.beta_r / (self.alpha_r - 1.0).clamp(min=EPSILON)
            K_tilde_diag = params['outputscale'] - torch.sum(A_batch * K_XZ_batch, dim=1)
            var_f_batch = E_r_inv * (K_tilde_diag + torch.sum((A_batch @ self.S_u) * A_batch, dim=1))
            
            # 3. Combine terms
            E_sq_err = (y_batch.squeeze() - mu_f_batch.squeeze())**2 + var_f_batch

            # --- Update local parameters ---
            alpha_lambda_batch = params['dof_lik'] / 2.0 + 0.5
            beta_lambda_batch = params['dof_lik'] / 2.0 + 0.5 * E_sq_err / params['noisescale']

            E_lambda_batch = alpha_lambda_batch / beta_lambda_batch.clamp(min=EPSILON)

            return alpha_lambda_batch, beta_lambda_batch, E_lambda_batch

    def _e_step_global(self, X_batch, y_batch, K_XZ_batch, K_ZZ, local_params, params, var_lr):
        # E-Step (Global)
        with torch.no_grad():
            _, _, E_lambda_batch = local_params
            scaling_factor = self.N / X_batch.shape[0]

            identity_M = torch.eye(self.M, device=self.device, dtype=X_batch.dtype)

            K_ZZ_inv = K_ZZ.solve(identity_M)
            K_ZZ_inv = to_linear_operator(K_ZZ_inv)
            A_batch = K_XZ_batch @ K_ZZ_inv
            
            # --- Target Parameters for q(u, r) ---
            # Target for q(u)
            E_r = self.alpha_r / self.beta_r.clamp(min=EPSILON)
            S_u_inv_data_term = (A_batch.T * E_lambda_batch) @ A_batch / params['noisescale'] * scaling_factor
            target_S_u_inv = E_r * K_ZZ_inv + S_u_inv_data_term
            target_S_u_inv = target_S_u_inv.add_jitter(JITTER)
            target_S_u = target_S_u_inv.solve(identity_M)
            m_u_data_term = A_batch.T @ torch.diag(E_lambda_batch) @ y_batch / params['noisescale'] * scaling_factor
            target_m_u = target_S_u @ m_u_data_term
            
            # Target for q(r) - using simplified update
            target_alpha_r = params['dof_func'] / 2.0 + self.M / 2.0
            E_quad_u = torch.trace(K_ZZ_inv @ (target_S_u + target_m_u @ target_m_u.T))
            target_beta_r = params['dof_func'] / 2.0 + E_quad_u / 2.0

            # Solve another variational problem argmin KL(q(u,r)||q(u)q(r))
            _, target_S_u, _, _ = get_optimal_gaussian_gamma(target_m_u, target_S_u, target_alpha_r, target_beta_r)

            # Convert current and target parameters to natural form for q(u, r)
            eta1_curr, eta2_curr, eta3_curr, eta4_curr = gaussian_gamma_standard_to_natural_covariance_param(self.m_u, self.S_u, self.alpha_r, self.beta_r)
            eta1_targ, eta2_targ, eta3_targ, eta4_targ = gaussian_gamma_standard_to_natural_covariance_param(target_m_u, target_S_u, target_alpha_r, target_beta_r)

            # Polyak averaging step
            eta1_new = (1 - var_lr) * eta1_curr + var_lr * eta1_targ
            eta2_new = (1 - var_lr) * eta2_curr + var_lr * eta2_targ
            eta3_new = (1 - var_lr) * eta3_curr + var_lr * eta3_targ
            eta4_new = (1 - var_lr) * eta4_curr + var_lr * eta4_targ

            # Convert back to standard parameters
            m_u_new, S_u_new, alpha_r_new, beta_r_new = gaussian_gamma_natural_to_standard_covariance_param(eta1_new, eta2_new, eta3_new, eta4_new)

            # Update model state
            self.m_u.data = m_u_new
            self.S_u.data = S_u_new
            self.alpha_r.data = alpha_r_new
            self.beta_r.data = beta_r_new

    def _m_step(self, optimizer, loss):
        if optimizer is None: return
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    def fit(
        self, 
        epochs=100, batch_size=128, 
        hyper_lr=0.01, var_lr=0.1,
        X_test=None, y_test=None, eval_interval=10
    ):
        parameters_to_optimize = [p for name, p in self.named_parameters() if self.hyper_optim_mode.get(name.replace("log_",""), "MLE") != 'FIX']
        
    
        optimizer = optim.Adam(parameters_to_optimize, lr=hyper_lr) if parameters_to_optimize else None
        dataset = TensorDataset(self.X_full, self.y_full)
        generator = torch.Generator(device='cpu')
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, generator=generator)

        history = {
            'elbo': [], 'log_prior': [], 'loss': [],
            'lengthscale': [], 'outputscale': [], 'dof_func': [], 'dof_lik': [], 'noisescale': [],
            'eval_epochs': [], 'eval_metrics': [], 'fit_times': []
        }
        logging.info(f"Starting SVI optimization for {epochs} epochs...")

        for epoch in range(epochs):
            for X_batch, y_batch in dataloader:

                fit_start_time = time.time()

                params = self._get_hyperparams()
                K_ZZ_base = to_linear_operator(self.kernel(self.Z, self.Z, params['lengthscale'], params['outputscale']))
                K_ZZ = K_ZZ_base.add_jitter(JITTER)
                K_XZ_batch = self.kernel(X_batch, self.Z, params['lengthscale'], params['outputscale'])

                # E-Step (Local)
                local_params = self._e_step_local(X_batch, y_batch, K_XZ_batch, K_ZZ, params)
                self._e_step_global(X_batch, y_batch, K_XZ_batch, K_ZZ, local_params, params, var_lr)

                # M-Step
                elbo = self._calculate_elbo(X_batch, y_batch, K_XZ_batch, K_ZZ, local_params)
                log_prior = self._calculate_log_prior(params)
                loss = - (elbo + log_prior)
                
                self._m_step(optimizer, loss)

                fit_end_time = time.time()

                # --- Store history ---
                params_final = self._get_hyperparams()
                history['elbo'].append(elbo.item())
                history['log_prior'].append(log_prior.item())
                history['loss'].append(loss.item())
                history['lengthscale'].append(params_final['lengthscale'].detach().cpu().numpy())
                history['outputscale'].append(params_final['outputscale'].item())
                history['noisescale'].append(params_final['noisescale'].item())
                history['dof_func'].append(params_final['dof_func'].item())
                history['dof_lik'].append(params_final['dof_lik'].item())
                history['fit_times'].append(fit_end_time - fit_start_time)

                
            if (epoch + 1) % eval_interval == 0:
                ls_str = ", ".join([f"{l:.3f}" for l in params_final['lengthscale']])
                logging.info(f"Epoch {epoch+1:3d}/{epochs} | Fit Time: {fit_end_time - fit_start_time:.3f}s | ELBO: {elbo.item():8.2f} | l: [{ls_str}] | var: {params_final['outputscale']:.3f} | noise2: {params_final['noisescale']:3f} | dof_func: {params_final['dof_func']:.2f} | dof_lik: {params_final['dof_lik']:.2f}")

            # --- Evaluation Step ---
            if X_test is not None and y_test is not None and (epoch + 1) % eval_interval == 0:
                metrics = self._evaluate(X_test, y_test)
                history['eval_epochs'].append(epoch + 1)
                history['eval_metrics'].append(metrics)
                logging.info(
                    f"Epoch {epoch+1:3d}/{epochs} | Test Metrics: "
                    f"RMSE: {metrics['rmse']:.3f}"
                )

        logging.info("Optimization finished.")
        return history

    def predict(self, X_test):
        """
        Calculates the parameters of the predictive distribution q(f_*) for new test data.
        The predictive distribution is a Student-t distribution.
        
        Args:
            X_test (torch.Tensor or np.ndarray): The test data points.
            
        Returns:
            dict: A dictionary containing the parameters of the Student-t distribution:
                  'loc' (mean), 'scale_sq' (scale squared), and 'dof' (degrees of freedom).
        """
        X_test = torch.as_tensor(X_test, dtype=self.X_full.dtype, device=self.device)
        with torch.no_grad():
            params = self._get_hyperparams()

            K_ZZ_base = to_linear_operator(self.kernel(self.Z, self.Z, params['lengthscale'], params['outputscale']))
            K_ZZ = K_ZZ_base.add_jitter(JITTER)
            K_star_Z = self.kernel(X_test, self.Z, params['lengthscale'], params['outputscale'])
            k_star_star = params['outputscale']
            
            identity_M = torch.eye(self.M, device=self.device, dtype=X_test.dtype)
            A_star = K_star_Z @ K_ZZ.solve(identity_M)
            
            # --- Parameters of the Student-t predictive distribution q(f_*) ---
            # Location (mean)
            mu_star = A_star @ self.m_u
            
            # Degrees of Freedom
            dof_star = 2 * self.alpha_r
            
            # Scale-squared
            scale_sq_star = (self.beta_r / self.alpha_r.clamp(min=EPSILON)) * \
                            (k_star_star - torch.sum(A_star * K_star_Z, dim=1) + torch.sum((A_star @ self.S_u) * A_star, dim=1))
            
            return {
                'loc': mu_star.squeeze(),
                'scale_sq': scale_sq_star.clamp(min=EPSILON),
                'dof': dof_star.clamp(min=EPSILON)
            }

    def _evaluate(self, X_test, y_test):
        """Evaluates the model on test data and returns a dictionary of metrics."""
        self.eval()
        with torch.no_grad():
            f_pred_tensor = self.predict(X_test)
            f_pred_numpy = f_pred_tensor['loc'].cpu().numpy()
            y_true_numpy = y_test.cpu().numpy()

            # metrics = {
            #     'rmse': np.sqrt(np.mean((y_true_numpy - f_pred_numpy)**2))
            # }
            metrics = {
                'rmse': np.sqrt(mean_squared_error(y_true_numpy, f_pred_numpy))
            }
        self.train()
        return metrics

